# Walkthrough: building a tool-using agent

This notebook builds a small ReAct agent over the calculator tool, using a stub LLM so it runs offline. Replace `ScriptedLLM` with `OpenAIClient(...)` to run live.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))
from agent_framework import AgentState, Tracer, ToolRegistry
from agent_framework.templates import build_react_agent
from agent_framework.tools import CalculatorTool
from agent_framework.llm.base import LLMClient, LLMResponse, LLMToolCall


In [ ]:
class ScriptedLLM(LLMClient):
    name = 'scripted'
    def __init__(self, script):
        self.script = list(script)
    def call(self, messages, tools=None, temperature=0.2, max_tokens=1024, response_model=None):
        return self.script.pop(0) if self.script else LLMResponse(text='done')

llm = ScriptedLLM([
    LLMResponse(text='', tool_calls=[LLMToolCall(id='1', name='calculator', args={'expression':'12*12'})]),
    LLMResponse(text='12*12 is 144', tool_calls=[]),
])
tools = ToolRegistry()
tools.register(CalculatorTool())

In [ ]:
tracer = Tracer(out_dir='../traces', backend='jsonl')
graph = build_react_agent(llm, tools)
compiled = graph.compile(tracer=tracer)
out = compiled.invoke(AgentState(input='what is 12 squared'))
out.output, out.tool_results

Open the dashboard with `make dashboard` and click into the trace ID printed below.

In [ ]:
print('trace id:', tracer.current_trace)